In [0]:
import requests

url = "https://api.adsb.lol/v2/all"
data = requests.get(url).json()


In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS bronze")

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType

schema = StructType([
    StructField("raw_json", StringType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

empty_df = spark.createDataFrame([], schema)

empty_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze.adsb_raw")

In [0]:
import requests
import json
from datetime import datetime
from pyspark.sql import Row

url = "https://api.adsb.lol/v2/all"

response = requests.get(url)
data = response.json()

row = Row(
    raw_json=json.dumps(data),
    ingestion_time=datetime.utcnow()
)

df = spark.createDataFrame([row])

df.write \
  .format("delta") \
  .mode("append") \
  .saveAsTable("bronze.adsb_raw")

print("Data Loaded Successfully")

In [0]:
bronze_df = spark.table("bronze.adsb_raw")

display(bronze_df)
